In [19]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import copy
from operator import itemgetter
import numpy as np
from pathlib import Path

def initially_preprocessed_data_frame():
    path_to_data = "/home/pc/moje/python_projects/adsorbents_isotherms/Data/all_data.csv"
    print(path_to_data)
    df = pd.read_csv(path_to_data)
    print("------------------------------- Preprocessing_data start -------------------------------")
    print(df.info())
    df['isotherm_X'] = df['isotherm_X'].apply(json.loads) # make it as float list
    df['isotherm_Y'] = df['isotherm_Y'].apply(json.loads) # make it as float list
    df['isotherm_X_Y'] = df['isotherm_X_Y'].apply(json.loads) # make it as float list
    df['axis_minx_maxx_miny_maxy']  = df['axis_minx_maxx_miny_maxy'].apply(json.loads) # make it as float list
    
    df['isotherm_X'] = df['isotherm_X'].apply(lambda x: [abs(num) for num in x]) # remove the minus sign
    df['isotherm_Y'] = df['isotherm_Y'].apply(lambda y: [abs(num) for num in y]) # remove the minus sign
    
    constant = 22.414
    df['isotherm_Y'] = df.apply( lambda row: [value * constant for value in row['isotherm_Y']] if row['Y_axis_type'] == 'mmol/g' else row['isotherm_Y'], axis=1 )
    df['isotherm_X_Y'] = df.apply( lambda row: [[value[0],(value[1] * constant)] for value in row['isotherm_X_Y']] if row['Y_axis_type'] == 'mmol/g' else row['isotherm_X_Y'], axis=1 )
    df['isotherm_X_Y'] = df.apply( lambda row: [[value[0],(value[1] * 1000)] for value in row['isotherm_X_Y']] if row['Y_axis_type'] == 'x1000' else row['isotherm_X_Y'], axis=1 )
    df['isotherm_X_Y'] = df.apply( lambda row: [[(value[0] / 100), value[1]] for value in row['isotherm_X_Y']] if row['Article_name'] == 'Data78' else row['isotherm_X_Y'], axis=1 )
    # print(df['isotherm_X_Y'] )

    # deleteing samples -------------
    df = df[~((df['Article_name'] == 'Data73') )]
    df = df[~((df['Article_name'] == 'Data15') )]
    df = df[~((df['Article_name'] == 'Data34') )]
    df = df[~((df['Article_name'] == 'Data7') & (df['Sample_name'] == 'RK-500')  & (df['Figure_number'] == '3'))]
    df = df[~((df['Article_name'] == 'Data44') & (df['Sample_name'] == 'UAK2-800d-without_washing')  & (df['Figure_number'] == '5'))]
    df = df[~((df['Article_name'] == 'Data94') & (df['Sample_name'] == 'UFFA-2-500')  & (df['Figure_number'] == '2_1'))]
    df = df[~((df['Article_name'] == 'Data23') & (df['Sample_name'] == '0%BO')  & (df['Figure_number'] == '2_2'))]
    df = df[~((df['Article_name'] == 'Data33') & (df['Sample_name'] == '600')  & (df['Figure_number'] == '2'))]
    df = df[~((df['Article_name'] == 'Data91') & (df['Sample_name'] == 'AC-MM5')  & (df['Figure_number'] == '3_3'))]
    df = df[~((df['Article_name'] == 'Data91') & (df['Sample_name'] == 'AC-MM7')  & (df['Figure_number'] == '3_3'))]
    df = df[~((df['Article_name'] == 'Data158') & (df['Sample_name'] == 'CS1000A')  & (df['Figure_number'] == '4'))]
    df = df[~((df['Article_name'] == 'Data158') & (df['Sample_name'] == 'CS1000AF')  & (df['Figure_number'] == '4'))]
    # deleteing samples -------------
    return df

def shape_isotherm(list_of_isotherm):
    x = [0.0000000000,
        0.0086206897,
        0.0172413793,
        0.0258620690,
        0.0344827586,
        0.0689655172,
        0.1034482759,
        0.1379310345,
        0.1724137931,
        0.2068965517,
        0.2413793103,
        0.2758620690,
        0.3103448276,
        0.3448275862,
        0.3793103448,
        0.4137931034,
        0.4482758621,
        0.4827586207,
        0.5172413793,
        0.5517241379,
        0.5862068966,
        0.6206896552,
        0.6551724138,
        0.6896551724,
        0.7241379310,
        0.7586206897,
        0.7931034483,
        0.8275862069,
        0.8620689655,
        0.8965517241,
        0.9310344828,
        0.9655172414,
        1.0000000000]
    shaped_isotherms = []
    for index , bananadata in enumerate(list_of_isotherm):
        one_iso_true_v2 = np.array(bananadata)
        x2 = one_iso_true_v2[:, 0]
        y2 = one_iso_true_v2[:, 1]
        
        data_do_podzialu_na_przedzialy_oś_x = x2
        data_do_podzialu_na_przedzialy_oś_y = y2
        data  = []
        counter = 0
        poprzednia = 0
        new_x = [] # końcowy zestaw x
        
        # tworzenie pierwszego zbioru y z wartościami nan 
        for liczba in x:
            if(liczba != 0.0 ):
                new_x.append(liczba)
                counter = counter+1
                suma = []
                for i in range(len(data_do_podzialu_na_przedzialy_oś_x)):
                    if(poprzednia<data_do_podzialu_na_przedzialy_oś_x[i]<liczba):
                        suma.append(data_do_podzialu_na_przedzialy_oś_y[i])
                data.append(np.mean(suma))   
                poprzednia = liczba
        
        # wypelnianie pojedzynczych nan, ale zostają grupki nan
        data = np.array(data)
        data = np.insert(data, 0, 0)
    
        if np.isnan(data[len(data)-1]): # add max value to last index if nan
            data[len(data)-1] = max(data)
    
        
        new_data = []
        for index, e in enumerate(data):
            if np.isnan(e) and (not np.isnan(data[index-1])) and (not np.isnan(data[index+1])):
                if index != 0:
                    if not np.isnan(data[index-1]):
                        index_przed = index - 1
                    if not np.isnan(data[index+1]):
                        index_po = index + 1
                        new_men = (np.mean( (data[index_przed] + data[index_po]) ) / 2) 
                        new_data.append(new_men)
            else:
                new_data.append(e)
        
        # wypelnianie grupek nan i uzyskanie gotowej ostatecznej listy danych
        poczatek_i_koniec_zbioru = []
        tymczasowe_mini_zbiory = []
        for index, e in enumerate(new_data):
            if len(poczatek_i_koniec_zbioru) == 0 :
                    if np.isnan(e) and (not np.isnan(data[index-1])) and (np.isnan(data[index+1])):
                        poczatek_i_koniec_zbioru.append(index)
            if len(poczatek_i_koniec_zbioru) == 1 :
                    if np.isnan(e):
                        tymczasowe_mini_zbiory.append(index)
                    if np.isnan(e) and (np.isnan(data[index-1])) and (not np.isnan(data[index+1])):
                        poczatek_i_koniec_zbioru.append(index)
    
        
            if len(poczatek_i_koniec_zbioru) == 2 :
                roznica = (new_data[poczatek_i_koniec_zbioru[1] + 1] - new_data[poczatek_i_koniec_zbioru[0] - 1])
                ile_dodac = roznica / (len(tymczasowe_mini_zbiory)+1)
                for inde in tymczasowe_mini_zbiory:
                    new_data[inde] = new_data[inde-1] + ile_dodac
                poczatek_i_koniec_zbioru = []
                tymczasowe_mini_zbiory = []
        
        new_x = np.array(new_x)
        new_x = np.insert(new_x, 0, 0)# Insert 0 at the beginning
        new_data = np.array(new_data) 
        # shaped_isotherms.append( [new_x, new_data] ) # new_x -> final x data, new_data -> final y data
        shaped_isotherms.append( new_data ) # new_x -> final x data, new_data -> final y data
    # shaped_isotherms = np.array(shaped_isotherms)
    return shaped_isotherms

def get_whole_preprocessed_dataframe():
    dff = initially_preprocessed_data_frame()
    x_y = dff['isotherm_X_Y'].to_list()
    X_Y_np_array = shape_isotherm(x_y)
    # izotherm_x_y_in_new_size_flatten = [xy.flatten() for xy in X_Y_np_array]
    # dff['processed_flatten_isotherm_X_Y'] = izotherm_x_y_in_new_size_flatten
    dff['processed_flatten_isotherm_X_Y'] = X_Y_np_array
    print(dff.info())
    print("------------------------------- Preprocessing_data end -------------------------------")
    return dff

# df  = get_whole_preprocessed_dataframe()


/home/pc/moje/python_projects/adsorbents_isotherms/Data/all_data.csv
------------------------------- Preprocessing_data start -------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 26 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Article_name                    1011 non-null   object 
 1   Figure_number                   1011 non-null   object 
 2   Sample_name                     1011 non-null   object 
 3   Total_surface_area[m2/g]        928 non-null    float64
 4   Total_pore_volume[cm3/g]        778 non-null    float64
 5   Micropore_volume[cm3/g]         730 non-null    float64
 6   Mesopore_volume[cm3/g]          21 non-null     float64
 7   Average_pore_diameter[nm]       121 non-null    float64
 8   Impregnation_ratio[agent/char]  20 non-null     float64
 9   Activation_type                 33 non-null     object 
 10

/home/pc/.cache/pypoetry/virtualenvs/adorbents-isotherms-FmcCsTt1-py3.12/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/pc/.cache/pypoetry/virtualenvs/adorbents-isotherms-FmcCsTt1-py3.12/lib/python3.12/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


<class 'pandas.core.frame.DataFrame'>
Index: 974 entries, 0 to 1010
Data columns (total 27 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Article_name                    974 non-null    object 
 1   Figure_number                   974 non-null    object 
 2   Sample_name                     974 non-null    object 
 3   Total_surface_area[m2/g]        892 non-null    float64
 4   Total_pore_volume[cm3/g]        744 non-null    float64
 5   Micropore_volume[cm3/g]         714 non-null    float64
 6   Mesopore_volume[cm3/g]          21 non-null     float64
 7   Average_pore_diameter[nm]       121 non-null    float64
 8   Impregnation_ratio[agent/char]  20 non-null     float64
 9   Activation_type                 33 non-null     object 
 10  Burn_off[%]                     75 non-null     float64
 11  Carbonization_time[h]           210 non-null    float64
 12  Carbonization_temperature[stC]  253 non-